# FashionMNIST Classifier — Walkthrough Notebook

This notebook collects everything built over the course of our conversation:

1. Load FashionMNIST with `torchvision` and split it into 55,000 train / 5,000 validation samples.
2. Preprocess images with `torchvision.transforms.v2`, casting to `torch.float32`.
3. Define a small CNN `ImageClassifier`.
4. Set up an optimizer, a `CrossEntropyLoss`, and a `torchmetrics.Accuracy` metric.
5. Train with a `train2` function that loops over epochs, logs a `history` dict, and validates each epoch via `evaluate_tm`.
6. Run predictions on a validation batch, check correctness, and inspect softmax probabilities (including top-4).
7. Count the total number of model parameters.

## 1–2. Imports, transforms, dataset loading, and train/val split

In [1]:
import torch
from torch.utils.data import random_split, DataLoader
import torchvision
import torchvision.transforms.v2 as transforms
import torchmetrics
import torch.nn.functional as F

# ---- Transform: convert PIL images to float32 tensors and normalize ----
transform = transforms.Compose([
    transforms.ToImage(),                              # PIL -> tv_tensor Image (uint8)
    transforms.ToDtype(torch.float32, scale=True),      # uint8 -> float32 in [0, 1]
    transforms.Normalize((0.2860,), (0.3530,))          # FashionMNIST mean/std
])

# ---- Download / load the full training set (60,000 images) ----
full_train_dataset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# ---- Load the official test set (10,000 images) ----
test_dataset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# ---- Split the 60,000 training images into 55,000 train / 5,000 val ----
train_size = 55_000
val_size = 5_000
assert train_size + val_size == len(full_train_dataset)

generator = torch.Generator().manual_seed(42)  # for reproducibility
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator
)

print(f"Full training set size: {len(full_train_dataset)}")
print(f"Train subset size:      {len(train_dataset)}")
print(f"Validation subset size: {len(val_dataset)}")
print(f"Test set size:          {len(test_dataset)}")

Full training set size: 60000
Train subset size:      55000
Validation subset size: 5000
Test set size:          10000


## DataLoaders

In [2]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print(f"Sample batch shape: {images.shape}")
print(f"Sample batch dtype:  {images.dtype}")
print(f"Sample labels shape: {labels.shape}")

Sample batch shape: torch.Size([64, 1, 28, 28])
Sample batch dtype:  torch.float32
Sample labels shape: torch.Size([64])


## 3. Model definition — a small CNN classifier

In [3]:
class ImageClassifier(torch.nn.Module):
    """Simple CNN for classifying 28x28 grayscale FashionMNIST images into 10 classes."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28x28 -> 28x28
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),                              # -> 14x14
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),  # -> 14x14
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),                              # -> 7x7
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 7 * 7, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ---- Instantiate model, move to device, define loss ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
xentropy = torch.nn.CrossEntropyLoss()

print(f"Device: {device}")
print(model)

Device: cuda
ImageClassifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


Quick sanity check: one forward pass and loss computation before any training.

In [4]:
images, labels = images.to(device), labels.to(device)
outputs = model(images)
loss = xentropy(outputs, labels)
print(f"Output shape: {outputs.shape}")
print(f"Initial loss (untrained): {loss.item():.4f}")

Output shape: torch.Size([64, 10])
Initial loss (untrained): 2.3169


## 4. Optimizer and accuracy metric

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

## 5. Training loop: `evaluate_tm` and `train2`

In [6]:
def evaluate_tm(model, loader, metric):
    """Run inference over a full loader and return the computed metric."""
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
           n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

Run training for a few epochs:

In [7]:
n_epochs = 5
history = train2(model, optimizer, xentropy, metric,
                  train_loader, val_loader, n_epochs)

Epoch 1/5, train loss: 0.4833, train metric: 0.8246, valid metric: 0.8814
Epoch 2/5, train loss: 0.3119, train metric: 0.8879, valid metric: 0.8960
Epoch 3/5, train loss: 0.2618, train metric: 0.9054, valid metric: 0.8924
Epoch 4/5, train loss: 0.2283, train metric: 0.9162, valid metric: 0.9058
Epoch 5/5, train loss: 0.2079, train metric: 0.9235, valid metric: 0.9146


## 6. Evaluation mode, predictions, and correctness check

In [8]:
# ---- Switch to evaluation mode ----
model.eval()

# ---- Extract a batch of evaluation (validation) data ----
eval_images, eval_labels = next(iter(val_loader))
eval_images, eval_labels = eval_images.to(device), eval_labels.to(device)

# ---- Get predicted y value: index of the largest logit ----
with torch.no_grad():
    eval_outputs = model(eval_images)
    predicted = eval_outputs.argmax(dim=1)

# ---- Print the class names at the predicted indexes ----
class_names = full_train_dataset.classes
predicted_labels = [class_names[idx] for idx in predicted.tolist()]
actual_labels = [class_names[idx] for idx in eval_labels.tolist()]

print("Predicted classes:", predicted_labels)
print("Actual classes:   ", actual_labels)

# ---- Check if predictions were correct ----
correct_mask = predicted == eval_labels
num_correct = correct_mask.sum().item()
print(f"Correct predictions: {num_correct}/{len(eval_labels)}")
print("Correct? ", correct_mask.tolist())

Predicted classes: ['Sneaker', 'Coat', 'Pullover', 'Sandal', 'Ankle boot', 'Bag', 'Sneaker', 'Sneaker', 'Sneaker', 'Coat', 'Dress', 'Coat', 'Sneaker', 'Sneaker', 'Shirt', 'T-shirt/top', 'Trouser', 'Dress', 'Sneaker', 'Trouser', 'Pullover', 'Coat', 'Dress', 'T-shirt/top', 'Coat', 'Pullover', 'Coat', 'Bag', 'Sandal', 'Sneaker', 'Shirt', 'Ankle boot', 'Coat', 'Dress', 'Sandal', 'Coat', 'Sandal', 'Ankle boot', 'Bag', 'Bag', 'Coat', 'Coat', 'T-shirt/top', 'Sneaker', 'Dress', 'Pullover', 'Pullover', 'Sandal', 'Ankle boot', 'T-shirt/top', 'Pullover', 'Ankle boot', 'Trouser', 'Bag', 'Ankle boot', 'Trouser', 'Shirt', 'Sneaker', 'Dress', 'Dress', 'Ankle boot', 'T-shirt/top', 'Ankle boot', 'Sandal']
Actual classes:    ['Sneaker', 'Coat', 'Pullover', 'Sandal', 'Ankle boot', 'Bag', 'Sneaker', 'Ankle boot', 'Sneaker', 'Coat', 'Dress', 'Coat', 'Sneaker', 'Sneaker', 'Shirt', 'T-shirt/top', 'Trouser', 'Dress', 'Sneaker', 'Trouser', 'Coat', 'Coat', 'T-shirt/top', 'Shirt', 'Shirt', 'Pullover', 'Coat', 'B

### Softmax probabilities for the predictions

In [9]:
# Convert logits to class probabilities
probabilities = F.softmax(eval_outputs, dim=1)

# Confidence of the predicted class for each sample
predicted_probs = probabilities.gather(1, predicted.unsqueeze(1)).squeeze(1)
print(predicted_probs)

tensor([0.9702, 0.9858, 0.9510, 1.0000, 1.0000, 1.0000, 1.0000, 0.9360, 0.9806,
        0.9452, 0.4955, 0.9339, 0.9993, 0.9977, 0.7663, 0.9999, 1.0000, 0.9934,
        0.9295, 1.0000, 0.5043, 0.9856, 0.2618, 0.7683, 0.7325, 0.9997, 0.9946,
        1.0000, 1.0000, 0.9035, 0.6053, 0.9994, 0.9975, 0.7111, 1.0000, 0.9997,
        0.9998, 0.9553, 1.0000, 1.0000, 0.9799, 0.6327, 0.9967, 1.0000, 0.9728,
        0.9959, 0.9998, 0.9976, 0.8291, 0.9992, 0.9970, 0.9508, 1.0000, 1.0000,
        0.9440, 0.9929, 0.9985, 0.9886, 0.6226, 0.9995, 0.9887, 0.9552, 1.0000,
        1.0000], device='cuda:0')


### Top-4 predicted classes and probabilities (rounded to 3 decimals)

In [10]:
top4_probs, top4_indexes = torch.topk(probabilities, k=4, dim=1)
top4_probs = torch.round(top4_probs * 1000) / 1000

print("Top-4 probabilities:\n", top4_probs)
print("Top-4 indexes:\n", top4_indexes)

Top-4 probabilities:
 tensor([[0.9700, 0.0290, 0.0010, 0.0000],
        [0.9860, 0.0130, 0.0010, 0.0000],
        [0.9510, 0.0340, 0.0120, 0.0020],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [0.9360, 0.0640, 0.0000, 0.0000],
        [0.9810, 0.0190, 0.0000, 0.0000],
        [0.9450, 0.0430, 0.0100, 0.0020],
        [0.4950, 0.4720, 0.0290, 0.0010],
        [0.9340, 0.0660, 0.0000, 0.0000],
        [0.9990, 0.0010, 0.0000, 0.0000],
        [0.9980, 0.0020, 0.0000, 0.0000],
        [0.7660, 0.2300, 0.0040, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [0.9930, 0.0040, 0.0020, 0.0000],
        [0.9300, 0.0690, 0.0010, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [0.5040, 0.4950, 0.0010, 0.0000],
        [0.9860, 0.0120, 0.0020, 0.0000],
        [0.2620, 0.2530, 0.1460, 0.1410],
        [0.7

## 7. Total number of model parameters

In [11]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 421,642
